# Valence Band Offset (VBO)

Calculate the valence band offset for a labeled interface using a DFT workflow on the Mat3ra platform.

<h2 style="color:green">Usage</h2>

1. Create and save an interface with labels (for example via `create_interface_with_min_strain_zsl.ipynb`).
1. Set the interface and calculation parameters in cells 1.2 and 1.3 below.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the VBO result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the interface, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create materials: load an interface from the `../uploads` folder, split it into interface/left/right parts using interface labels, strip labels required by Quantum ESPRESSO, and save all three materials to the platform in a materials set.
1. Configure workflow: select application, load the VBO workflow from Standata, assign materials to subworkflows by role, set model and computational parameters, and preview the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with materials and workflow configuration: assemble the job from materials, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the valence band offset.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
INTERFACE_NAME = "Interface"
LEFT_SIDE_PART = InterfacePartsEnum.SUBSTRATE
RIGHT_SIDE_PART = InterfacePartsEnum.FILM
MATERIALS_SET_NAME = None  # Defaults to the loaded interface name

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "valence_band_offset"
MY_WORKFLOW_NAME = "VBO"

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

### 1.3. Set specific VBO parameters


In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # "us" (ultrasoft), "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"  # for gga: "pbe", "pbesol"; for lda: "pz"
MODEL_SUBTYPE = "gga"

# K-grid and k-path
SCF_KGRID = None  # e.g. [8, 8, 1]
KPATH = None  # e.g. [{"point": "G", "steps": 20}, {"point": "M", "steps": 20}]

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".


In [ ]:
import os

os.environ["API_PORT"] = "3000"
os.environ["API_SECURE"] = "false"
os.environ["API_HOST"] = "localhost"
os.environ["OIDC_ACCESS_TOKEN"] = "Q-9wDdNt-QJwA5Q6XAZTU1wNdFfoEambJEw1kBa7K9z"

In [ ]:
from utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Create materials
### 3.1. Load interface from local folder


In [ ]:
from utils.jupyterlite import load_material_from_folder
from utils.visualize import visualize_materials as visualize

interface = load_material_from_folder(FOLDER, INTERFACE_NAME)

visualize(interface, repetitions=[1, 1, 1])
visualize(interface, repetitions=[1, 1, 1], rotation="-90x")

### 3.2. Create materials from interface parts
Slabs are isolated based on labels, then labels removed as they are not compatible with Quantum ESPRESSO. The three materials (interface, left slab, right slab) are named and visualized.

In [ ]:
from mat3ra.made.tools.modify import interface_get_part

interface_system_name = MATERIALS_SET_NAME or interface.name
left_material = interface_get_part(interface, part=LEFT_SIDE_PART)
right_material = interface_get_part(interface, part=RIGHT_SIDE_PART)
interface_material = interface.clone()

for material in [left_material, right_material]:
    material.basis.set_labels_from_list([])
    material.name = f"{material.formula} Slab"
interface_material.basis.set_labels_from_list([])
interface_material.name = interface_system_name

materials_by_role = {"interface": interface_material, "left": left_material, "right": right_material}
for role, material in materials_by_role.items():
    print(f"  {role}: {material.name}")
visualize(list(materials_by_role.values()), repetitions=[1, 1, 1])


### 3.3. Save materials and add to a materials set


In [ ]:
from mat3ra.made.material import Material
from utils.api import get_or_create_material

existing = client.materials.list({"name": interface_system_name, "owner._id": ACCOUNT_ID})
materials_set = next((m for m in existing if m.get("isSet")), None)
if materials_set is None:
    materials_set = client.materials.create_set({"name": interface_system_name, "owner": {"_id": ACCOUNT_ID}})
print(f"✅ Materials set: {materials_set['name']} ({materials_set['_id']})")

saved_materials = {}
for role, material in materials_by_role.items():
    saved = Material.create(get_or_create_material(client, material, ACCOUNT_ID))
    client.materials.move_to_set(saved.id, "", materials_set["_id"])
    saved_materials[role] = saved
    print(f"  {role}: {saved.name} ({saved.id})")


## 4. Configure workflow
### 4.1. Select application


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 4.2. Load workflow from Standata and preview it


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)


### 4.3. Set model and its parameters (physics)


In [ ]:
from mat3ra.mode.model import Model
from mat3ra.standata.model_tree import ModelTreeStandata

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = Model.create(model_config)

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name == APPLICATION_NAME:
        subworkflow.model = model


### 4.4. Modify method (computational parameters): k-grid, k-path, and cutoffs


In [ ]:
from mat3ra.wode.context.providers import (
    PlanewaveCutoffsContextProvider,
    PointsGridDataProvider,
    PointsPathDataProvider,
)

for subworkflow in workflow.subworkflows:
    if subworkflow.application.name != APPLICATION_NAME:
        continue

    unit_names = [unit.name for unit in subworkflow.units]

    if SCF_KGRID is not None and "pw_scf" in unit_names:
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
        subworkflow.set_unit(unit)

    if KPATH is not None and "pw_bands" in unit_names:
        unit = subworkflow.get_unit_by_name(name="pw_bands")
        unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).yield_data())
        subworkflow.set_unit(unit)

    cutoffs_context = PlanewaveCutoffsContextProvider(
        wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True
    ).yield_data()
    for unit_name in ["pw_scf", "pw_bands"]:
        if unit_name not in unit_names:
            continue
        unit = subworkflow.get_unit_by_name(name=unit_name)
        unit.add_context(cutoffs_context)
        subworkflow.set_unit(unit)


### 4.5. Preview final workflow


In [ ]:
visualize_workflow(workflow)


## 5. Create the compute configuration
### 5.1. Get list of clusters


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 5.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 6. Create the job
### 6.1. Create job


In [ ]:
from utils.api import create_job
from utils.generic import dict_to_namespace
from utils.visualize import display_JSON

materials = list(saved_materials.values())
job_name = f"{MY_WORKFLOW_NAME} {interface_system_name} {timestamp}"
workflow.name = job_name

print(f"Materials: {[m.id for m in materials]}")
print(f"Project: {project_id}")

job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")

display_JSON(job_response)


## 7. Submit the job and monitor the status


In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")


In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve and visualize results
### 8.1. Valence Band Offset


In [ ]:
from mat3ra.prode import PropertyName
from utils.visualize import visualize_properties

vbo_value = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.valence_band_offset.value)[0]
print(f"Valence Band Offset (VBO) value: {vbo_value['value']:.3f} eV")

vbo_data = client.properties.get_for_job(job_id)
visualize_properties(vbo_data, title="Valence Band Offset")